In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("data/final-articles.csv")

In [2]:
# standardizing column names
df = df.rename(columns={"publish-time": "publish_time"})

In [3]:
# creating a function to chunk passed in text
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0

    n = len(words)
    while start < n:
        end = min(start + chunk_size, n)
        chunk = words[start:end]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap

    return chunks

In [4]:
all_chunks = []
for i, row in df.iterrows():
    full_text = f"TITLE: {row['title']}\n\n{row['content']}"

    chunks = chunk_text(full_text)

    for j, chunk in enumerate(chunks):
        all_chunks.append({
            "doc_id": i,
            "chunk_id": f"{i}_{j}",
            "text": chunk,
            "title": row["title"]
        })



chunks_df = pd.DataFrame(all_chunks)

In [5]:
print(len(chunks_df))
print(chunks_df["text"].str.len().describe())
print(chunks_df.sample(1)["text"].iloc[0])

33982
count    33982.000000
mean      1383.234683
std        534.081746
min          2.000000
25%       1019.000000
50%       1680.000000
75%       1754.000000
max       2467.000000
Name: text, dtype: float64
TITLE: Queen Elizabeth lauds 'inspirational' England after Euro 2022 win Queen Elizabeth II joined in celebrations following England's Euro 2022 final win on Sunday with a letter celebrating the Lionesses' historic triumph. England lifted the title for the first time with victory over Germany in extra-time, Chloe Kelly striking late to score the decisive goal. The game had finished 1-1 at full time, with Ella Toone and Lena Magull both hitting in the second half at Wembley. "My warmest congratulations, and those of my family, go to you all on winning the European Women’s Football Championships," read the letter, signed Elizabeth R. “Your success goes far beyond the trophy you have so deservedly earned.“You have all set an example that will be an inspiration for girls and women tod

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1062 [00:00<?, ?it/s]

In [10]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name = "soccer_articles"
)

In [18]:
chunk_ids = chunks_df["chunk_id"].astype(str).tolist()
BATCH_SIZE = 500

for start in range(0, len(chunks_df), BATCH_SIZE):
    end = start + BATCH_SIZE

    collection.add(
        ids=chunk_ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist()
    )

    print(f"Added {min(end, len(chunks_df))}/{len(chunks_df)}")

Added 500/33982
Added 1000/33982
Added 1500/33982
Added 2000/33982
Added 2500/33982
Added 3000/33982
Added 3500/33982
Added 4000/33982
Added 4500/33982
Added 5000/33982
Added 5500/33982
Added 6000/33982
Added 6500/33982
Added 7000/33982
Added 7500/33982
Added 8000/33982
Added 8500/33982
Added 9000/33982
Added 9500/33982
Added 10000/33982
Added 10500/33982
Added 11000/33982
Added 11500/33982
Added 12000/33982
Added 12500/33982
Added 13000/33982
Added 13500/33982
Added 14000/33982
Added 14500/33982
Added 15000/33982
Added 15500/33982
Added 16000/33982
Added 16500/33982
Added 17000/33982
Added 17500/33982
Added 18000/33982
Added 18500/33982
Added 19000/33982
Added 19500/33982
Added 20000/33982
Added 20500/33982
Added 21000/33982
Added 21500/33982
Added 22000/33982
Added 22500/33982
Added 23000/33982
Added 23500/33982
Added 24000/33982
Added 24500/33982
Added 25000/33982
Added 25500/33982
Added 26000/33982
Added 26500/33982
Added 27000/33982
Added 27500/33982
Added 28000/33982
Added 28500/

In [32]:
query = "Manchester United tactical approach"

query_embedding = model.encode(query)



In [33]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results = 5
)

In [34]:
for i, doc in enumerate(results["documents"][0]):
    print(f"\n=== Result {i+1} ===")
    print(doc[:1000])


=== Result 1 ===
lines of four then waiting to improvise on the counter. It is the simplest tactical strategy to coach – the easiest way to compress space – but the United manager has now abandoned the one tactical element that was working. It has led directly to the concession of nine goals in the two Premier League games United have played against last year’s top five. “I think, for example, you look at the first goal, it can’t be possible that they can have sort of three running through in the first five minutes,” Shaw said of Naby Keita’s fifth-minute opener. “We need to be more compact, we need to be better and we know that.” Put simply, the depth of tactical detail in the modern game looks like it’s currently beyond Solskjaer’s reach. It requires years of tactical education to understand and coach the complexities of a unified press, or of the ‘automatisms’ required to enact structured and progressive possession. You cannot simply tell your players to press harder, to express th